# Automatic Music Transcription
* Audio to guitar tablature

# TODO
* Consider using an embedding layer at the input, as it can help in learning a more meaningful representation of the tokens


In [84]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Preprocessing

In [3]:
# Load in the mp3 file, get the sample rate, and get the data
# from the file
#

import numpy as np
import matplotlib.pyplot as plt
import librosa

# Load in the mp3 file
y, sr = librosa.load('dataset/john-mayer/3_Why_Georgia.mp3')

In [4]:
# Get the data from the file
data = librosa.feature.mfcc(y=y, sr=sr)

In [375]:
def plotMFCC(data):
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(data, x_axis='time')
    plt.colorbar()
    plt.title('MFCC')
    plt.tight_layout()
    plt.show()

# plotMFCC(data)

In [7]:
# install pyguitarpro
# !pip install pyguitarpro
# !pip install --upgrade attrs


In [9]:
import guitarpro

# Load in the gp5 file
song = guitarpro.parse('dataset/john-mayer/3_why-georgia.gp5')

In [377]:
# token_note_format = 'instrument:note:string:fret'
# token_rest_format = 'instrument:rest'

# song.tempo, song.key

In [378]:
def tokenize_note_effect(note_effect):
    effect_tokens = []
    
    if note_effect.accentuatedNote:
        effect_tokens.append('accentuated')
        
    if note_effect.bend:
        effect_tokens.append(f'bend:{note_effect.bend.type}')
        
    if note_effect.ghostNote:
        effect_tokens.append('ghost')
        
    if note_effect.grace:
        effect_tokens.append(f'grace:{note_effect.grace.fret}:{note_effect.grace.duration}:{note_effect.grace.transition}')
        
    if note_effect.hammer:
        effect_tokens.append('hammer')
        
    if note_effect.harmonic:
        effect_tokens.append(f'harmonic:{type(note_effect.harmonic).__name__}')
        
    if note_effect.heavyAccentuatedNote:
        effect_tokens.append('heavy_accentuated')
        
    if note_effect.leftHandFinger != 'open':
        effect_tokens.append(f'left_finger:{note_effect.leftHandFinger}')
        
    if note_effect.letRing:
        effect_tokens.append('let_ring')
        
    if note_effect.palmMute:
        effect_tokens.append('palm_mute')
        
    if note_effect.rightHandFinger != 'open':
        effect_tokens.append(f'right_finger:{note_effect.rightHandFinger}')
        
    if note_effect.slides:
        for slide in note_effect.slides:
            effect_tokens.append(f'slide:{slide}')
            
    if note_effect.staccato:
        effect_tokens.append('staccato')
        
    if note_effect.tremoloPicking:
        effect_tokens.append(f'tremolo:{note_effect.tremoloPicking.duration.value}')
        
    if note_effect.trill:
        effect_tokens.append(f'trill:{note_effect.trill.fret}:{note_effect.trill.duration.value}')
        
    if note_effect.vibrato:
        effect_tokens.append('vibrato')
        
    # Join the tokens with a comma and return the result
    effect_token = ','.join(effect_tokens)
    return effect_token


In [380]:
def tokenize_note(note):
    token = f'note:{note.string}:{note.value}'

    if note.type.name == 'dead':
        # append 'dead' to the token
        token += ':dead'
        return token

    if isinstance(note.effect, guitarpro.NoteEffect):  # This part is pseudo-code; adapt based on actual PyGuitarPro attributes
        note_effect_token = tokenize_note_effect(note.effect)
        token += f':nfx:{note_effect_token}'
    
    return token

In [392]:
def tokenize_song(song):
    tokens = []
    
    # Start tokens
    tokens.append(f'artist:{song.artist}')
    tokens.append(f'downtune:{song.tracks[0].strings[0].value}')  # Assuming first track, first string for downtune info
    tokens.append(f'tempo:{song.tempo}')
    tokens.append('start')
    tokens.append('new_measure')

    for track in song.tracks:
        for measure in track.measures:
            for voice in measure.voices:
                for beat in voice.beats:
                    if track.isPercussionTrack:
                        pass    # TODO Skip drums for now
                        # tokens.append(f'drums:note:{beat.notes[0].value}')  # Assuming one note per beat for drums
                    else:
                        # Note tokens for pitched instruments (guitar, bass, etc. not drums)
                        for note in beat.notes:
                            token = tokenize_note(note)
                            tokens.append(token)
                    
                    # Assuming 'wait' tokens are in terms of beats; adapt as needed
                    tokens.append(f'wait:{beat.duration.value}')

                    # Rest tokens
                    if beat.status == guitarpro.BeatStatus.rest:
                        tokens.append('note:rest')
                        
                    # Add any beat effects, note effects, tempo changes, etc.
                    # ... (this part depends on what effects you want to include)
    tokens.append('end')
    return tokens

# Load the song using PyGuitarPro (pseudo-code)
# song = guitarpro.parse('path/to/your/file.gp5')

# Tokenize the song
# tokenize_song(song)

# Print the tokens (or save them, etc.)
# print(tokens)


In [35]:
import copy

# set acoustic_song to song but copy the value so we don't change the original
acoustic_song = copy.deepcopy(song)
acoustic_song.tracks = [acoustic_song.tracks[0]]
print(len(song.tracks), len(acoustic_song.tracks))

In [350]:
def count_acoustic_notes():
    acoustic_notes = []
    max_num_notes = 5
    counter = 0
    
    for track in acoustic_song.tracks:
        for measure in track.measures:
            for voice in measure.voices:
                for beat in voice.beats:
                    for note in beat.notes:
                        acoustic_notes.append(note)

                        counter += 1
                        if counter > max_num_notes:
                            return acoustic_notes

acoustic_notes = count_acoustic_notes()

In [402]:
# len(acoustic_notes), acoustic_notes[:5], tokenize_note(acoustic_notes[:5])
# for note in acoustic_notes[:5]:
#     print(note)
#     print(tokenize_note(note))
#     print()

dir(acoustic_notes[0].effect)

['__annotations__',
 '__attrs_attrs__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slotnames__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'accentuatedNote',
 'bend',
 'ghostNote',
 'grace',
 'hammer',
 'harmonic',
 'heavyAccentuatedNote',
 'isBend',
 'isDefault',
 'isFingering',
 'isGrace',
 'isHarmonic',
 'isTremoloPicking',
 'isTrill',
 'leftHandFinger',
 'letRing',
 'palmMute',
 'rightHandFinger',
 'slides',
 'staccato',
 'tremoloPicking',
 'trill',
 'vibrato']

In [396]:
tokens = tokenize_song(acoustic_song)

In [397]:
tokens[:1000]

['artist:John Mayer',
 'downtune:64',
 'tempo:98',
 'start',
 'new_measure',
 'wait:4',
 'Guitar 1:note:rest',
 'note:4:0:dead',
 'note:5:0:dead',
 'note:6:3:nfx:left_finger:Fingering.open,let_ring,right_finger:Fingering.open',
 'wait:8',
 'note:2:3:nfx:left_finger:Fingering.open,right_finger:Fingering.open',
 'note:3:0:nfx:hammer,left_finger:Fingering.open,right_finger:Fingering.open',
 'wait:16',
 'note:3:2:nfx:left_finger:Fingering.open,right_finger:Fingering.open',
 'wait:16',
 'note:5:0:dead',
 'note:6:0:dead',
 'wait:16',
 'note:2:3:nfx:left_finger:Fingering.open,let_ring,right_finger:Fingering.open',
 'note:3:0:nfx:left_finger:Fingering.open,let_ring,right_finger:Fingering.open',
 'wait:16',
 'note:5:0:nfx:left_finger:Fingering.open,let_ring,right_finger:Fingering.open',
 'wait:8',
 'note:4:4:nfx:left_finger:Fingering.open,let_ring,right_finger:Fingering.open',
 'note:5:5:nfx:left_finger:Fingering.open,let_ring,right_finger:Fingering.open',
 'wait:8',
 'note:3:0:nfx:left_finger:

In [384]:
# Initialize an empty dictionary to store the mapping from tokens to integers
token_to_int = {}
int_to_token = {}
next_int = 0  # Start assigning from 0

# Function to dynamically update the dictionary
def update_token_mapping(token, token_to_int, int_to_token, next_int):
    if token not in token_to_int:
        token_to_int[token] = next_int
        int_to_token[next_int] = token
        next_int += 1
    return next_int

# Update the dictionary with the tokens from the file
for token in tokens:
    next_int = update_token_mapping(token, token_to_int, int_to_token, next_int)

# Show some statistics and samples to verify
num_unique_tokens = len(token_to_int)
sample_items_token_to_int = list(token_to_int.items())[:45]
sample_items_int_to_token = list(int_to_token.items())[:45]

num_unique_tokens, sample_items_token_to_int, sample_items_int_to_token


(51,
 [('artist:John Mayer', 0),
  ('downtune:64', 1),
  ('tempo:98', 2),
  ('start', 3),
  ('new_measure', 4),
  ('wait:4', 5),
  ('Guitar 1:note:rest', 6),
  ('Guitar 1:note:4:0', 7),
  ('Guitar 1:note:5:0', 8),
  ('Guitar 1:note:6:3', 9),
  ('nfx:left_finger:Fingering.open,let_ring,right_finger:Fingering.open', 10),
  ('wait:8', 11),
  ('Guitar 1:note:2:3', 12),
  ('nfx:left_finger:Fingering.open,right_finger:Fingering.open', 13),
  ('Guitar 1:note:3:0', 14),
  ('nfx:hammer,left_finger:Fingering.open,right_finger:Fingering.open', 15),
  ('wait:16', 16),
  ('Guitar 1:note:3:2', 17),
  ('Guitar 1:note:6:0', 18),
  ('Guitar 1:note:4:4', 19),
  ('Guitar 1:note:5:5', 20),
  ('Guitar 1:note:2:0', 21),
  ('Guitar 1:note:5:3', 22),
  ('Guitar 1:note:3:4', 23),
  ('Guitar 1:note:4:2', 24),
  ('Guitar 1:note:4:5', 25),
  ('Guitar 1:note:1:0', 26),
  ('Guitar 1:note:5:2', 27),
  ('Guitar 1:note:6:2', 28),
  ('Guitar 1:note:1:3', 29),
  ('Guitar 1:note:2:1', 30),
  ('Guitar 1:note:4:3', 31),
  

In [123]:
# num_unique_tokens, sample_items_token_to_int, sample_items_int_to_token
sample_items_token_to_int

[('artist:John Mayer', 0),
 ('downtune:64', 1),
 ('tempo:98', 2),
 ('start', 3),
 ('new_measure', 4),
 ('wait:4', 5),
 ('Guitar 1:note:rest', 6),
 ('Guitar 1:note:s4:f0:type:dead', 7),
 ('Guitar 1:note:s5:f0:type:dead', 8),
 ('Guitar 1:note:s6:f3:type:normal', 9),
 ("nfx:['nfx:left_finger:Fingering.open', 'nfx:let_ring', 'nfx:right_finger:Fingering.open']",
  10),
 ('wait:8', 11),
 ('Guitar 1:note:s2:f3:type:normal', 12),
 ("nfx:['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open']",
  13),
 ('Guitar 1:note:s3:f0:type:normal', 14),
 ("nfx:['nfx:hammer', 'nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open']",
  15),
 ('wait:16', 16),
 ('Guitar 1:note:s3:f2:type:normal', 17),
 ('Guitar 1:note:s6:f0:type:dead', 18),
 ('Guitar 1:note:s5:f0:type:normal', 19),
 ('Guitar 1:note:s4:f4:type:normal', 20),
 ('Guitar 1:note:s5:f5:type:normal', 21),
 ('Guitar 1:note:s2:f0:type:normal', 22),
 ('Guitar 1:note:s5:f3:type:normal', 23),
 ('Guitar 1:note:s3:f4:type:normal',

In [129]:
# Convert the tokens to their corresponding integer values using the mapping
integer_representation = [token_to_int[token] for token in tokens]

# Show some sample integer representation of the tokens
integer_representation[:10], integer_representation[-10:]


([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], [13, 14, 13, 36, 13, 8, 9, 13, 75, 76])

In [325]:
len(integer_representation)

8621

In [131]:
len(integer_representation)

8621

In [132]:
max(integer_representation)

76

In [136]:
# print the first 25 tokens along with their integer representations
for i in range(50):
    print('{:4} {:20} {}'.format(i, tokens[i], integer_representation[i]))

   0 artist:John Mayer    0
   1 downtune:64          1
   2 tempo:98             2
   3 start                3
   4 new_measure          4
   5 wait:4               5
   6 Guitar 1:note:rest   6
   7 Guitar 1:note:s4:f0:type:dead 7
   8 Guitar 1:note:s5:f0:type:dead 8
   9 Guitar 1:note:s6:f3:type:normal 9
  10 nfx:['nfx:left_finger:Fingering.open', 'nfx:let_ring', 'nfx:right_finger:Fingering.open'] 10
  11 wait:8               11
  12 Guitar 1:note:s2:f3:type:normal 12
  13 nfx:['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open'] 13
  14 Guitar 1:note:s3:f0:type:normal 14
  15 nfx:['nfx:hammer', 'nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open'] 15
  16 wait:16              16
  17 Guitar 1:note:s3:f2:type:normal 17
  18 nfx:['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open'] 13
  19 wait:16              16
  20 Guitar 1:note:s5:f0:type:dead 8
  21 Guitar 1:note:s6:f0:type:dead 18
  22 wait:16              16
  23 Guitar 1:note:s2:

In [139]:
# convert the integer representation back to the original tokens
reconstructed_tokens = [int_to_token[i] for i in integer_representation]

# print the first 25 tokens along with their integer representations
for i in range(50):
    print('{:4} {:20} {}'.format(i, reconstructed_tokens[i], integer_representation[i]))


   0 artist:John Mayer    0
   1 downtune:64          1
   2 tempo:98             2
   3 start                3
   4 new_measure          4
   5 wait:4               5
   6 Guitar 1:note:rest   6
   7 Guitar 1:note:s4:f0:type:dead 7
   8 Guitar 1:note:s5:f0:type:dead 8
   9 Guitar 1:note:s6:f3:type:normal 9
  10 nfx:['nfx:left_finger:Fingering.open', 'nfx:let_ring', 'nfx:right_finger:Fingering.open'] 10
  11 wait:8               11
  12 Guitar 1:note:s2:f3:type:normal 12
  13 nfx:['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open'] 13
  14 Guitar 1:note:s3:f0:type:normal 14
  15 nfx:['nfx:hammer', 'nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open'] 15
  16 wait:16              16
  17 Guitar 1:note:s3:f2:type:normal 17
  18 nfx:['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open'] 13
  19 wait:16              16
  20 Guitar 1:note:s5:f0:type:dead 8
  21 Guitar 1:note:s6:f0:type:dead 18
  22 wait:16              16
  23 Guitar 1:note:s2:

## Audio Segmentation
* Split the audio into segments of roughly the same length as the tablature measures.

In [170]:
len(y)

5931006

In [315]:
# Given parameters
sample_rate = sr
chunk_duration = 2.5  # Chunk duration in seconds
hop_length = 512  # This is the default hop length for librosa's mfcc calculation

# Calculate the number of samples per chunk
samples_per_chunk = int(sample_rate * chunk_duration)

# Calculate the number of chunks
num_chunks = len(y) // samples_per_chunk + 1

# calculate total number of frames
total_frames = len(y) // hop_length + 1

# Calculate the number of frames per chunk
# Formula: Number of Frames = (Number of Samples in Segment - Frame Length) / Hop Length + 1
# Assuming the frame length is equal to the hop length for simplicity
frames_per_chunk = (samples_per_chunk) // hop_length + 1

In [316]:
# Assuming 'y' is your original audio waveform
audio_segments = []
# let start be at the 0.5 second mark
start = int(0.5 * sample_rate)  
for start_frame in range(start, len(y), frames_per_chunk * hop_length):
    end_frame = start_frame + frames_per_chunk * hop_length
    segment = y[start_frame:end_frame]
    audio_segments.append(segment)

# 'audio_segments' now contains chunks of the original audio waveform

In [317]:
# play the first audio segment 
import IPython.display as ipd
ipd.Audio(audio_segments[0], rate=sample_rate)


In [318]:
# play the second audio segment
ipd.Audio(audio_segments[1], rate=sample_rate)

In [319]:
ipd.Audio(audio_segments[2], rate=sample_rate)

In [320]:
ipd.Audio(audio_segments[3], rate=sample_rate)

In [321]:
ipd.Audio(audio_segments[4], rate=sample_rate)

In [322]:
ipd.Audio(audio_segments[5], rate=sample_rate)

In [323]:
ipd.Audio(audio_segments[6], rate=sample_rate)

In [324]:
ipd.Audio(audio_segments[7], rate=sample_rate)

In [339]:
# count the number of measures in the gp5 file
num_measures = 0
for track in song.tracks:
    for measure in track.measures:
        num_measures += 1
    print(f'{track.name} measures: {num_measures}')


Guitar 1 measures: 104
Guitar 2 measures: 208
Guitar 3 measures: 312
Guitar 4 measures: 416
Bass measures: 520
Drums measures: 624


In [338]:
104 * 2.5

260.0

In [345]:
def tokenize_song_by_measure(song):
    measures_tokens = []  # This will be a list of lists, where each sublist represents tokens for a measure
    
    # Initialize tokens for the song metadata
    song_tokens = [
        f'artist:{song.artist}',
        f'downtune:{song.tracks[0].strings[0].value}',  # Assuming first track, first string for downtune info
        f'tempo:{song.tempo}',
        'start'
    ]
    
    for track in song.tracks:
        for measure in track.measures:
            measure_tokens = ['new_measure']  # Start a new measure
            
            for voice in measure.voices:
                for beat in voice.beats:
                    if track.isPercussionTrack:
                        # TODO: Handle percussion separately
                        pass
                    else:
                        # Note tokens for pitched instruments
                        for note in beat.notes:
                            note_token = f'{track.name}:note:s{note.string}:f{note.value}:type:{note.type.name}'
                            measure_tokens.append(note_token)
                            
                            # Add note effects if any
                            if note.effect:
                                note_effect_token = tokenize_note_effect(note.effect)  # Assuming this function is defined elsewhere
                                measure_tokens.append(f'nfx:{note_effect_token}')
                    
                    # Add wait tokens based on beat duration
                    measure_tokens.append(f'wait:{beat.duration.value}')
                    
                    # Add rest tokens if the beat is a rest
                    if beat.status == guitarpro.BeatStatus.rest:
                        measure_tokens.append(f'{track.name}:note:rest')
                        
                    # Add any other effects here
                    
            # Add the measure's tokens to the main list
            measures_tokens.append(measure_tokens)
    
    # Append the song metadata at the beginning of the first measure
    measures_tokens[0] = song_tokens + measures_tokens[0]
    measures_tokens.append(['end'])  # Append the 'end' token after the last measure
    
    return measures_tokens

# Load the song using PyGuitarPro (pseudo-code)
# song = guitarpro.parse('path/to/your/file.gp5')

# Tokenize the song by measure
measures_tokens = tokenize_song_by_measure(acoustic_song)

# Print the tokens for each measure (or save them, etc.)
for i, measure in enumerate(measures_tokens):
    print(f'Measure {i+1}: {measure}')

# The rest of your code for creating the token_to_int and int_to_token mappings can follow here,
# as well as converting each measure's tokens to integers.


Measure 1: ['artist:John Mayer', 'downtune:64', 'tempo:98', 'start', 'new_measure', 'wait:4', 'Guitar 1:note:rest']
Measure 2: ['new_measure', 'Guitar 1:note:s4:f0:type:dead', "nfx:['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open']", 'Guitar 1:note:s5:f0:type:dead', "nfx:['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open']", 'Guitar 1:note:s6:f3:type:normal', "nfx:['nfx:left_finger:Fingering.open', 'nfx:let_ring', 'nfx:right_finger:Fingering.open']", 'wait:8', 'Guitar 1:note:s2:f3:type:normal', "nfx:['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open']", 'Guitar 1:note:s3:f0:type:normal', "nfx:['nfx:hammer', 'nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open']", 'wait:16', 'Guitar 1:note:s3:f2:type:normal', "nfx:['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open']", 'wait:16', 'Guitar 1:note:s5:f0:type:dead', "nfx:['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open']", 'Guitar 1:note:s6

In [347]:
len(measures_tokens), len(measures_tokens[0])

(105, 7)